In [3]:
# Install missing dependencies for Jupyter kernel
%pip install geopandas matplotlib pandas


Defaulting to user installation because normal site-packages is not writeable
  Using cached geopandas-1.1.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
Using cached geopandas-1.1.4-py3-none-any.whl (343 kB)
Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl (23.8 MB)
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   -------- ------------------------------- 1.3/6.3 MB 13.5 MB/s eta 0:00:01
   --------------------- ------------------ 3.4/6.3 MB 10.3 MB/s eta 0:00:01
   -------------------------------------- - 6.0/6.3 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 6.3/6.3 MB 11.0 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 11.1 MB/s  0:00:00

   ---------------------------------------- 0/4 [shapely]
   ---------------------------------------- 0/4 [shapely]
   ----------------------------

02_plot_cluster_map.py

Tugas Orang 5 — Peta persebaran klaster FCM per provinsi.

Input (data real dari Orang 1 & 4):
- data/external/indonesia_38_provinces.geojson
- data/interim/fcm_membership_geojson.csv   ← output script 01
    Kolom kunci: province_name_geojson, crisp_cluster, cluster_label

Output:
- outputs/figures/peta_klaster_provinsi.png

Desain:
- Klaster 1 (risiko relatif rendah) : biru  #2563EB
- Klaster 2 (risiko relatif tinggi) : merah #DC2626
- Tidak ada data (Papua Tengah & Pegunungan) : abu-abu #CCCCCC
- Border: putih tipis antar-provinsi, hitam tebal untuk batas luar peta
- Anotasi: nama klaster + jumlah provinsi di legenda

In [4]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

PROJECT_ROOT = Path().resolve().parent
GEOJSON_PATH    = PROJECT_ROOT / "data" / "external" / "indonesia_38_provinces.geojson"
HARMONIZED_PATH = PROJECT_ROOT / "data" / "interim" / "fcm_membership_geojson.csv"
OUTPUT_PATH     = PROJECT_ROOT / "outputs" / "figures" / "peta_klaster_provinsi.png"

# Warna klaster (indeks = crisp_cluster - 1)
CLUSTER_COLORS = {
    1: "#2563EB",  # biru  — Profil risiko relatif lebih rendah
    2: "#DC2626",  # merah — Profil risiko relatif lebih tinggi
}
NO_DATA_COLOR  = "#CCCCCC"
EDGE_COLOR_MAP = "#FFFFFF"
EDGE_LW_MAP    = 0.5

In [5]:
def main():
    gdf = gpd.read_file(GEOJSON_PATH)
    df  = pd.read_csv(HARMONIZED_PATH)

    # Spatial join
    merged = gdf.merge(
        df[["province_name_geojson", "crisp_cluster", "cluster_label"]],
        left_on="PROVINSI",
        right_on="province_name_geojson",
        how="left",
    )

    no_data = merged[merged["crisp_cluster"].isna()]
    if not no_data.empty:
        print(f"[INFO] {len(no_data)} provinsi tanpa data FCM (abu-abu): "
              f"{no_data['PROVINSI'].tolist()}")

    merged["plot_color"] = merged["crisp_cluster"].map(CLUSTER_COLORS).fillna(NO_DATA_COLOR)

    # ── Gambar peta ───────────────────────────────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(16, 8))
    fig.patch.set_facecolor("white")

    merged.plot(
        color=merged["plot_color"],
        edgecolor=EDGE_COLOR_MAP,
        linewidth=EDGE_LW_MAP,
        ax=ax,
    )

    # ── Legenda ───────────────────────────────────────────────────────────────
    n_by_cluster = df.groupby("crisp_cluster").size()
    handles = []
    labels_map = {
        1: "Klaster 1 — Profil risiko\nfaktor relatif lebih rendah",
        2: "Klaster 2 — Profil risiko\nfaktor relatif lebih tinggi",
    }
    for c, color in CLUSTER_COLORS.items():
        n = n_by_cluster.get(c, 0)
        handles.append(mpatches.Patch(color=color, label=f"{labels_map[c]}\n({n} provinsi)"))
    handles.append(mpatches.Patch(color=NO_DATA_COLOR, label="Tidak ada data FCM\n(Papua Tengah & Pegunungan)"))

    legend = ax.legend(
        handles=handles,
        loc="lower left",
        fontsize=9,
        frameon=True,
        framealpha=0.92,
        title="Klaster FCM (c=2, m=1.5)",
        title_fontsize=9.5,
        borderpad=0.8,
    )
    legend.get_frame().set_linewidth(0.8)

    ax.set_title(
        "Peta Persebaran Klaster Risiko Stunting per Provinsi\n"
        "Fuzzy C-Means Clustering (c=2, m=1.5) — Data SSGI 2022",
        fontsize=14,
        weight="bold",
        pad=14,
    )
    ax.axis("off")
    fig.tight_layout(pad=1.5)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_PATH, dpi=220, bbox_inches="tight", facecolor="white")
    print(f"Peta klaster disimpan → {OUTPUT_PATH}")
    plt.close(fig)

### Jalankan Pipeline

In [6]:
if __name__ == "__main__":
    main()

[INFO] 2 provinsi tanpa data FCM (abu-abu): ['Papua Tengah', 'Papua Pegunungan']
Peta klaster disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\figures\peta_klaster_provinsi.png
